## Reproduction notebook setup

See `docs/notebooks.md` for inputs, execution order, and environment setup.
Run a working copy prepared by `tools/prepare_workspace.py`.
Saved outputs were removed from this copy; executed originals remain on the NAS.
The following cell changes file locations only. The analysis formulas are preserved.


In [ ]:
import os
from pathlib import Path

# Set these before launching Jupyter when using another storage mount.
VQ_NAS_ROOT = Path(os.environ.get("VQ_NAS_ROOT", "/megaverse/storage/nasiri"))
VQ_JUPYTER_ROOT = Path(os.environ.get("VQ_JUPYTER_ROOT", str(VQ_NAS_ROOT / "jupyter")))


# Chapter 9 preliminary extension: prefix budgets under rendition shift

Chapter 8 showed that a dataset can produce a large shift in token use without producing worse full-budget reconstruction scores. This preliminary extension asks whether the reconstruction comparison changes when the decoder receives only a prefix of an ordered token sequence. It evaluates two recent ordered tokenizers, FlexTok and One-D-Piece, rather than the fixed-grid tokenizers used in the main thesis experiments.

The main question is:

> **How does rendition shift change the relationship between prefix length and reconstruction quality?**

We answer it at three levels:

1. Does the absolute ImageNet versus ImageNet-R comparison change with token budget?
2. At the planned 32-token endpoint, does ImageNet-R lose more quality relative to its own 256-token reconstruction?
3. In an exploratory check, has ImageNet-R completed a smaller fraction of its available 8-to-256-token improvement at the same prefix length?

The third question separates a change in curve shape from a larger absolute range of improvement. Positive, negative, and unresolved results are all informative. None of these measurements identifies semantic information in an individual token or isolates a causal ordering mechanism.

## Exact setup

Both models use the same 10,000 ImageNet images and the same 10,000 ImageNet-R images: 50 images from each of 200 shared classes in each domain. The domains share class labels, but not scenes. Each image is converted to RGB, resized on the shorter edge, and center-cropped to 256 by 256 pixels. The tested prefixes contain 8, 16, 32, 64, 96, 128, 160, 208, or 256 tokens.

The released checkpoints are `EPFL-VILAB/flextok_d12_d12_in1k` and `turing-motors/One-D-Piece-S-256`. FlexTok uses 25 decoding steps, guidance 15, and APG, following Appendix E.5, Table 8 of the FlexTok paper. Its recorded batch-derived noise seed is reused across prefix lengths. One-D-Piece is deterministic. Equal token counts do not imply equal bit rates or compute.

LPIPS is the primary metric because the 32-token LPIPS comparison was specified before this focused analysis. Lower LPIPS is better. PSNR and SSIM remain supporting checks because they respond differently to image content. All summaries first average images within a class and then weight the 200 classes equally. Pointwise 95% intervals use 5,000 shared class-bootstrap resamples. They do not cover new domains, repeated FlexTok decoder seeds, or all budgets simultaneously.

The first two analyses are confirmatory descriptions of the planned endpoint and full curves. The relative-progress analysis is exploratory. It was added after inspecting the absolute and endpoint-adjusted results.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Image as DisplayImage, Markdown

SEED = 0
BOOTSTRAP_REPLICATES = 5_000
MODEL_ORDER = ["FlexTok", "One-D-Piece"]
DATASET_ORDER = ["imagenet", "imagenet_r"]
DATASET_LABELS = {"imagenet": "ImageNet", "imagenet_r": "ImageNet-R"}
DATASET_COLORS = {"imagenet": "#4F6D7A", "imagenet_r": "#C05A3D"}
PREFIX_ORDER = [8, 16, 32, 64, 96, 128, 160, 208, 256]
METRICS = ["lpips", "psnr", "ssim"]
METRIC_LABELS = {
    "lpips": "LPIPS (lower is better)",
    "psnr": "PSNR (dB; higher is better)",
    "ssim": "SSIM (higher is better)",
}

NOTEBOOK_DIR = Path.cwd().resolve()
assert NOTEBOOK_DIR.name == "notebooks", NOTEBOOK_DIR
OUTPUT_DIR = NOTEBOOK_DIR / "chapter9_outputs"
RAW_DIR = OUTPUT_DIR / "data" / "raw"
DERIVED_DIR = OUTPUT_DIR / "data" / "derived" / "focused_v3"
FIGURE_DIR = OUTPUT_DIR / "figures" / "focused_v3"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RUN_FILES = {
    ("FlexTok", "imagenet"): "flextok_imagenet_per_image_metrics.csv",
    ("FlexTok", "imagenet_r"): "flextok_imagenet_r_per_image_metrics.csv",
    ("One-D-Piece", "imagenet"): "one_d_piece_imagenet_per_image_metrics.csv",
    ("One-D-Piece", "imagenet_r"): "one_d_piece_imagenet_r_per_image_metrics.csv",
}
USECOLS = ["model", "dataset", "wnid", "image", "prefix_tokens",
           "decode_seconds", "lpips", "psnr", "ssim"]

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": .22,
})

def save_table(frame, name):
    path = DERIVED_DIR / name
    frame.to_csv(path, index=False)
    return path

def save_figure(figure, name):
    path = FIGURE_DIR / name
    figure.savefig(path, bbox_inches="tight")
    return path

In [ ]:
# Load the four saved runs and form the exact common image cohort.
frames = []
inventory = []
for model in MODEL_ORDER:
    for dataset in DATASET_ORDER:
        frame = pd.read_csv(RAW_DIR / RUN_FILES[(model, dataset)], usecols=USECOLS)
        frame["model_label"] = model
        frame["dataset_key"] = dataset
        frame["prefix_tokens"] = frame.prefix_tokens.astype(int)
        assert not frame.duplicated(["image", "prefix_tokens"]).any()
        assert np.isfinite(frame[METRICS + ["decode_seconds"]].to_numpy()).all()
        schedule_ok = frame.groupby("image").prefix_tokens.apply(
            lambda values: set(values) == set(PREFIX_ORDER)
        ).all()
        assert schedule_ok
        inventory.append({
            "model": model, "dataset": DATASET_LABELS[dataset],
            "images_in_export": frame.image.nunique(),
            "classes_in_export": frame.wnid.nunique(),
        })
        frames.append(frame)
metrics_df = pd.concat(frames, ignore_index=True)

IMAGE_KEY = ["dataset_key", "wnid", "image"]
selected = metrics_df.loc[metrics_df.model_label == "FlexTok", IMAGE_KEY].drop_duplicates()
counts = selected.groupby(["dataset_key", "wnid"]).size()
assert len(selected) == 20_000 and len(counts) == 400 and counts.eq(50).all()
WNIDS = sorted(selected.loc[selected.dataset_key == "imagenet", "wnid"].unique())
assert len(WNIDS) == 200
assert WNIDS == sorted(selected.loc[selected.dataset_key == "imagenet_r", "wnid"].unique())

matched = []
for model in MODEL_ORDER:
    available = metrics_df[metrics_df.model_label == model]
    coverage = selected.merge(available[IMAGE_KEY].drop_duplicates(), on=IMAGE_KEY,
                              how="left", indicator=True, validate="one_to_one")
    assert coverage._merge.eq("both").all()
    model_frame = available.merge(selected, on=IMAGE_KEY, how="inner", validate="many_to_one")
    assert len(model_frame) == 20_000 * len(PREFIX_ORDER)
    matched.append(model_frame)
analysis_df = pd.concat(matched, ignore_index=True)
scope_df = analysis_df.groupby(["model_label", "dataset_key"]).agg(
    images=("image", "nunique"), classes=("wnid", "nunique"), rows=("image", "size")
).reset_index()
assert scope_df.images.eq(10_000).all() and scope_df.classes.eq(200).all()
save_table(scope_df, "ch09_comparison_scope.csv")
display(scope_df.rename(columns={"model_label": "model", "dataset_key": "dataset"}))

# Reuse the same sampled class indices for every model, domain, metric, and budget.
BOOT_INDEX = np.random.default_rng(SEED).integers(
    0, len(WNIDS), size=(BOOTSTRAP_REPLICATES, len(WNIDS))
)

def bootstrap_means(matrix):
    matrix = np.asarray(matrix, dtype=float)
    assert matrix.shape[0] == len(WNIDS) and np.isfinite(matrix).all()
    flat = matrix.reshape(len(WNIDS), -1)
    draws = np.empty((BOOTSTRAP_REPLICATES, flat.shape[1]))
    for start in range(0, BOOTSTRAP_REPLICATES, 64):
        indices = BOOT_INDEX[start:start + 64]
        draws[start:start + len(indices)] = flat[indices].mean(axis=1)
    return flat.mean(axis=0), draws

def paired_intervals(matrix):
    point, draws = bootstrap_means(matrix)
    lower, upper = np.quantile(draws, [.025, .975], axis=0)
    return point, lower, upper

In [ ]:
# Class-balanced absolute quality and truncation cost relative to 256 tokens.
class_absolute_df = analysis_df.groupby(
    ["model_label", "dataset_key", "wnid", "prefix_tokens"], as_index=False
)[METRICS].mean()

absolute_gap_rows = []
for model in MODEL_ORDER:
    for metric in METRICS:
        wide = class_absolute_df[class_absolute_df.model_label == model].pivot(
            index="wnid", columns=["dataset_key", "prefix_tokens"], values=metric
        ).reindex(index=WNIDS)
        matrix = np.column_stack([
            wide[("imagenet_r", prefix)].to_numpy() - wide[("imagenet", prefix)].to_numpy()
            for prefix in PREFIX_ORDER
        ])
        mean, lower, upper = paired_intervals(matrix)
        for j, prefix in enumerate(PREFIX_ORDER):
            absolute_gap_rows.append({
                "model": model, "metric": metric, "prefix_tokens": prefix,
                "mean_r_minus_imagenet": mean[j],
                "ci95_lower": lower[j], "ci95_upper": upper[j],
            })
absolute_gap_df = pd.DataFrame(absolute_gap_rows)

full = analysis_df[analysis_df.prefix_tokens == 256][
    ["model_label", "dataset_key", "wnid", "image"] + METRICS
].rename(columns={metric: f"{metric}_full" for metric in METRICS})
penalty_df = analysis_df.merge(
    full, on=["model_label", "dataset_key", "wnid", "image"],
    how="left", validate="many_to_one"
)
penalty_df["lpips_penalty"] = penalty_df.lpips - penalty_df.lpips_full
penalty_df["psnr_penalty"] = penalty_df.psnr_full - penalty_df.psnr
penalty_df["ssim_penalty"] = penalty_df.ssim_full - penalty_df.ssim
PENALTY_COLS = [f"{metric}_penalty" for metric in METRICS]
class_penalty = penalty_df.groupby(
    ["model_label", "dataset_key", "wnid", "prefix_tokens"], as_index=False
)[PENALTY_COLS].mean()

id_class = class_penalty[class_penalty.dataset_key == "imagenet"].drop(columns="dataset_key")
ood_class = class_penalty[class_penalty.dataset_key == "imagenet_r"].drop(columns="dataset_key")
class_delta = id_class.merge(
    ood_class, on=["model_label", "wnid", "prefix_tokens"],
    suffixes=("_imagenet", "_imagenet_r"), validate="one_to_one"
)
for metric in METRICS:
    class_delta[f"{metric}_delta"] = (
        class_delta[f"{metric}_penalty_imagenet_r"] -
        class_delta[f"{metric}_penalty_imagenet"]
    )
assert np.allclose(class_delta.loc[class_delta.prefix_tokens == 256,
                                   [f"{m}_delta" for m in METRICS]], 0)

bootstrap_rows = []
contrast_rows = []
for metric in METRICS:
    for model in MODEL_ORDER:
        matrix = class_delta[class_delta.model_label == model].pivot(
            index="wnid", columns="prefix_tokens", values=f"{metric}_delta"
        ).reindex(index=WNIDS, columns=PREFIX_ORDER)
        mean, lower, upper = paired_intervals(matrix.to_numpy())
        for j, prefix in enumerate(PREFIX_ORDER):
            bootstrap_rows.append({
                "model": model, "metric": metric, "prefix_tokens": prefix,
                "mean_ood_minus_imagenet": mean[j],
                "ci95_lower": lower[j], "ci95_upper": upper[j],
                "positive_class_fraction": (matrix[prefix] > 0).mean(),
            })
    wide = class_delta.pivot(
        index=["wnid", "prefix_tokens"], columns="model_label", values=f"{metric}_delta"
    ).reset_index()
    for prefix in PREFIX_ORDER:
        values = wide[wide.prefix_tokens == prefix].set_index("wnid").reindex(WNIDS)
        difference = values["FlexTok"].to_numpy() - values["One-D-Piece"].to_numpy()
        mean, lower, upper = paired_intervals(difference[:, None])
        contrast_rows.append({
            "kind": "endpoint_adjusted_domain_delta", "metric": metric,
            "prefix_tokens": prefix, "flex_minus_one_d": mean[0],
            "ci95_lower": lower[0], "ci95_upper": upper[0],
        })
bootstrap_df = pd.DataFrame(bootstrap_rows)
contrast_df = pd.DataFrame(contrast_rows)

save_table(class_absolute_df, "ch09_class_absolute_quality.csv")
save_table(absolute_gap_df, "ch09_absolute_domain_gaps.csv")
save_table(class_delta, "ch09_class_truncation_differences.csv")
save_table(bootstrap_df, "ch09_truncation_bootstrap_intervals.csv")
save_table(contrast_df, "ch09_model_contrasts.csv")

# Inputs required by the qualitative grid.
RECON_DIR = RAW_DIR / "saved_reconstructions_v1"
recon_manifest = pd.read_csv(RECON_DIR / "manifest.csv")
recon_examples = recon_manifest[recon_manifest.prefix_tokens == 0].copy()
assert not recon_manifest.duplicated(["model", "dataset", "image", "prefix_tokens"]).any()
metric_lookup = analysis_df.set_index(
    ["model_label", "dataset_key", "wnid", "image", "prefix_tokens"]
)

## 1. Absolute reconstruction quality

This is the direct continuation of Chapter 8. Each point is the equal-weight mean of 200 class means. Error bars and ribbons are pointwise 95% class-bootstrap intervals. ImageNet and ImageNet-R use the same resampled classes. The token axis uses a log2 scale because the budgets are multiplicative and unevenly spaced. Within each metric, the two model rows share one y scale.

The key comparison is the vertical distance between the domain curves. The final cell reports direct class-paired intervals for that distance at 32 and 256 tokens.

In [ ]:
# Class-bootstrap intervals for absolute reconstruction quality.
absolute_ci_rows = []
for model in MODEL_ORDER:
    for dataset in DATASET_ORDER:
        for metric in METRICS:
            matrix = class_absolute_df[
                (class_absolute_df.model_label == model) &
                (class_absolute_df.dataset_key == dataset)
            ].pivot(index="wnid", columns="prefix_tokens", values=metric).reindex(
                index=WNIDS, columns=PREFIX_ORDER
            )
            mean, lower, upper = paired_intervals(matrix.to_numpy())
            for j, prefix in enumerate(PREFIX_ORDER):
                absolute_ci_rows.append({
                    "model": model, "dataset": dataset, "metric": metric,
                    "prefix_tokens": prefix, "mean": mean[j],
                    "ci95_lower": lower[j], "ci95_upper": upper[j],
                })
absolute_ci = pd.DataFrame(absolute_ci_rows)
save_table(absolute_ci, "ch09_absolute_quality_bootstrap_intervals.csv")

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey="col")
for row, model in enumerate(MODEL_ORDER):
    for col, metric in enumerate(METRICS):
        ax = axes[row, col]
        for dataset in DATASET_ORDER:
            sub = absolute_ci[(absolute_ci.model == model) &
                              (absolute_ci.dataset == dataset) &
                              (absolute_ci.metric == metric)].sort_values("prefix_tokens")
            x = sub.prefix_tokens.to_numpy()
            y = sub["mean"].to_numpy()
            lo = sub.ci95_lower.to_numpy()
            hi = sub.ci95_upper.to_numpy()
            color = DATASET_COLORS[dataset]
            ax.fill_between(x, lo, hi, color=color, alpha=.13, linewidth=0)
            ax.errorbar(x, y, yerr=[y - lo, hi - y], color=color, marker="o",
                        linewidth=2, elinewidth=1, capsize=2,
                        label=DATASET_LABELS[dataset])
        ax.set_xscale("log", base=2)
        ax.set_xticks(PREFIX_ORDER, [str(x) for x in PREFIX_ORDER], rotation=35)
        ax.set_title(f"{model} | {metric.upper()}")
        ax.set_xlabel("retained tokens, log2 scale")
        ax.set_ylabel(METRIC_LABELS[metric])
axes[0, 0].legend(frameon=False)
fig.suptitle("Class-balanced reconstruction quality with 95% intervals", fontsize=15, y=1.02)
fig.tight_layout()
save_figure(fig, "ch09_fig_absolute_prefix_curves_rigorous.png")
plt.close(fig)
display(DisplayImage(filename=str(FIGURE_DIR / "ch09_fig_absolute_prefix_curves_rigorous.png")))

**How to read this figure.** A shifted dataset is not automatically harder to reconstruct. At 32 tokens, the mean LPIPS point estimate is worse on ImageNet-R for both systems, although the direct One-D-Piece interval includes zero. At 128 and 256 tokens, ImageNet-R has better mean LPIPS. PSNR and SSIM give different domain orderings. The result is therefore about a relationship among domain, budget, and metric rather than a universal out-of-distribution penalty.

## 2. Exploratory check: relative progress through the curve

The endpoint-adjusted truncation cost is measured in raw LPIPS units. A domain with a larger total improvement between 8 and 256 tokens has more LPIPS to lose. This check asks a different question: by budget (k), what fraction of that domain's available 8-to-256-token improvement has been achieved?

\[
F_D(k)=\frac{\bar L_D(8)-\bar L_D(k)}{\bar L_D(8)-\bar L_D(256)}.
\]

The ratio is recomputed inside every shared class-bootstrap resample. Zero is fixed at 8 tokens and one at 256 tokens. A negative ImageNet-R minus ImageNet difference means that ImageNet-R has completed less of its observed improvement at the same prefix. This is a normalized reconstruction-curve measure, not a percentage of semantic information recovered.

In [ ]:
# Recompute the complete ratio inside every class-bootstrap draw.
progress_rows = []
progress_delta_rows = []
denominator_rows = []
for model in MODEL_ORDER:
    domain_results = {}
    for dataset in DATASET_ORDER:
        matrix = class_absolute_df[
            (class_absolute_df.model_label == model) &
            (class_absolute_df.dataset_key == dataset)
        ].pivot(index="wnid", columns="prefix_tokens", values="lpips").reindex(
            index=WNIDS, columns=PREFIX_ORDER
        )
        point_loss, draw_loss = bootstrap_means(matrix.to_numpy())
        point_denominator = point_loss[0] - point_loss[-1]
        draw_denominator = draw_loss[:, 0] - draw_loss[:, -1]
        assert point_denominator > 0 and np.all(draw_denominator > 0)
        point_progress = (point_loss[0] - point_loss) / point_denominator
        draw_progress = (draw_loss[:, [0]] - draw_loss) / draw_denominator[:, None]
        lower, upper = np.quantile(draw_progress, [.025, .975], axis=0)
        domain_results[dataset] = (point_progress, draw_progress)
        denominator_rows.append({
            "model": model, "dataset": dataset,
            "lpips_improvement_8_to_256": point_denominator,
            "bootstrap_min": draw_denominator.min(),
            "bootstrap_max": draw_denominator.max(),
        })
        for j, prefix in enumerate(PREFIX_ORDER):
            progress_rows.append({
                "model": model, "dataset": dataset, "prefix_tokens": prefix,
                "fraction_achieved": point_progress[j],
                "ci95_lower": lower[j], "ci95_upper": upper[j],
            })
    point_delta = domain_results["imagenet_r"][0] - domain_results["imagenet"][0]
    draw_delta = domain_results["imagenet_r"][1] - domain_results["imagenet"][1]
    lower, upper = np.quantile(draw_delta, [.025, .975], axis=0)
    for j, prefix in enumerate(PREFIX_ORDER):
        progress_delta_rows.append({
            "model": model, "prefix_tokens": prefix,
            "imagenet_r_minus_imagenet": point_delta[j],
            "ci95_lower": lower[j], "ci95_upper": upper[j],
        })

progress_df = pd.DataFrame(progress_rows)
progress_delta_df = pd.DataFrame(progress_delta_rows)
denominator_df = pd.DataFrame(denominator_rows)
save_table(progress_df, "ch09_relative_progress_intervals.csv")
save_table(progress_delta_df, "ch09_relative_progress_domain_differences.csv")
save_table(denominator_df, "ch09_relative_progress_denominator_check.csv")

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True, sharey=True)
for ax, model in zip(axes, MODEL_ORDER):
    for dataset in DATASET_ORDER:
        sub = progress_df[(progress_df.model == model) &
                          (progress_df.dataset == dataset)].sort_values("prefix_tokens")
        x = sub.prefix_tokens.to_numpy()
        y = sub.fraction_achieved.to_numpy()
        lo = sub.ci95_lower.to_numpy()
        hi = sub.ci95_upper.to_numpy()
        color = DATASET_COLORS[dataset]
        ax.fill_between(x, lo, hi, color=color, alpha=.13, linewidth=0)
        ax.errorbar(x, y, yerr=[y - lo, hi - y], color=color, marker="o",
                    linewidth=2, elinewidth=1, capsize=2, label=DATASET_LABELS[dataset])
    ax.axvline(32, color="#333333", linestyle="--", linewidth=.8)
    ax.set_xscale("log", base=2)
    ax.set_xticks(PREFIX_ORDER, [str(x) for x in PREFIX_ORDER], rotation=35)
    ax.set_ylim(-.03, 1.03)
    ax.set_title(model)
    ax.set_ylabel("fraction of 8-to-256 LPIPS improvement")
    ax.legend(frameon=False)
axes[-1].set_xlabel("retained tokens, log2 scale")
fig.suptitle("Relative progress through the LPIPS curve", fontsize=15, y=1.01)
fig.tight_layout()
save_figure(fig, "ch09_fig_relative_lpips_progress.png")
plt.close(fig)
display(DisplayImage(filename=str(FIGURE_DIR / "ch09_fig_relative_lpips_progress.png")))

rows = []
for model in MODEL_ORDER:
    values = progress_df[(progress_df.model == model) &
                         (progress_df.prefix_tokens == 32)].set_index("dataset")
    delta = progress_delta_df[(progress_delta_df.model == model) &
                              (progress_delta_df.prefix_tokens == 32)].iloc[0]
    rows.append(
        f"| {model} | {values.loc['imagenet', 'fraction_achieved']:.1%} | "
        f"{values.loc['imagenet_r', 'fraction_achieved']:.1%} | "
        f"{delta.imagenet_r_minus_imagenet:+.1%} "
        f"[{delta.ci95_lower:+.1%}, {delta.ci95_upper:+.1%}] |"
    )
display(Markdown("""At 32 tokens:

| Model | ImageNet progress | ImageNet-R progress | ImageNet-R minus ImageNet, 95% interval |
| --- | ---: | ---: | ---: |
""" + "\n".join(rows)))

**Status of this check.** This normalization was chosen after the raw truncation result was known, so it is exploratory. Its denominator uses the observed 8- and 256-token endpoints. The bootstrap check above requires every sampled denominator to remain positive. The final cell interprets the 32-token difference without treating a nonzero estimate as a causal property of token ordering.

## 3. Truncation cost relative to full reconstruction

For each image, truncation cost compares a prefix with that image's 256-token reconstruction. The two domain curves appear on the same axes with pointwise 95% class-bootstrap intervals. The shaded area between their means is the endpoint-adjusted domain difference. Orange means ImageNet-R pays the larger cost. Gray means ImageNet does.

The thick connector marks the planned 32-token endpoint. The two model rows share one y scale within each metric. This is an absolute metric difference, unlike the relative-progress ratio above.

In [ ]:
# Bootstrap each domain's truncation cost, then show their vertical separation.
penalty_ci_rows = []
for model in MODEL_ORDER:
    for dataset in DATASET_ORDER:
        for metric in METRICS:
            matrix = class_penalty[(class_penalty.model_label == model) &
                                   (class_penalty.dataset_key == dataset)].pivot(
                index="wnid", columns="prefix_tokens", values=f"{metric}_penalty"
            ).reindex(index=WNIDS, columns=PREFIX_ORDER)
            mean, lower, upper = paired_intervals(matrix.to_numpy())
            for j, prefix in enumerate(PREFIX_ORDER):
                penalty_ci_rows.append({
                    "model": model, "dataset": dataset, "metric": metric,
                    "prefix_tokens": prefix, "mean": mean[j],
                    "ci95_lower": lower[j], "ci95_upper": upper[j],
                })
penalty_ci = pd.DataFrame(penalty_ci_rows)
save_table(penalty_ci, "ch09_domain_truncation_cost_bootstrap_intervals.csv")

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey="col")
for row, model in enumerate(MODEL_ORDER):
    for col, metric in enumerate(METRICS):
        ax = axes[row, col]
        domain_means = {}
        for dataset in DATASET_ORDER:
            sub = penalty_ci[(penalty_ci.model == model) &
                             (penalty_ci.dataset == dataset) &
                             (penalty_ci.metric == metric)].sort_values("prefix_tokens")
            x = sub.prefix_tokens.to_numpy()
            y = sub["mean"].to_numpy()
            lo = sub.ci95_lower.to_numpy()
            hi = sub.ci95_upper.to_numpy()
            color = DATASET_COLORS[dataset]
            domain_means[dataset] = y
            ax.fill_between(x, lo, hi, color=color, alpha=.12, linewidth=0)
            ax.errorbar(x, y, yerr=[y - lo, hi - y], color=color, marker="o",
                        linewidth=2, elinewidth=1, capsize=2,
                        label=DATASET_LABELS[dataset])
        image_net = domain_means["imagenet"]
        image_net_r = domain_means["imagenet_r"]
        ax.fill_between(PREFIX_ORDER, image_net, image_net_r,
                        where=image_net_r >= image_net, interpolate=True,
                        color=DATASET_COLORS["imagenet_r"], alpha=.16,
                        label="ImageNet-R extra cost")
        ax.fill_between(PREFIX_ORDER, image_net, image_net_r,
                        where=image_net_r < image_net, interpolate=True,
                        color="#777777", alpha=.13,
                        label="ImageNet extra cost")
        p32 = PREFIX_ORDER.index(32)
        ax.vlines(32, image_net[p32], image_net_r[p32], color="#111111", linewidth=3)
        ax.annotate("adjusted difference at 32", (32, (image_net[p32] + image_net_r[p32]) / 2),
                    xytext=(5, 0), textcoords="offset points", fontsize=8, va="center")
        ax.axhline(0, color="#333333", linestyle="--", linewidth=.8)
        ax.set_xscale("log", base=2)
        ax.set_xticks(PREFIX_ORDER, [str(x) for x in PREFIX_ORDER], rotation=35)
        ax.set_title(f"{model} | {metric.upper()}")
        ax.set_xlabel("retained tokens, log2 scale")
        ax.set_ylabel("cost relative to 256 tokens")
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle("Domain-specific truncation cost and the gap between domains", fontsize=15, y=1.02)
fig.tight_layout()
save_figure(fig, "ch09_fig_domain_truncation_cost_overlay.png")
plt.close(fig)
display(DisplayImage(filename=str(FIGURE_DIR / "ch09_fig_domain_truncation_cost_overlay.png")))

**How to read this figure.** The vertical gap at 32 tokens is positive for LPIPS in both systems. That result means ImageNet-R loses more LPIPS relative to its own full reconstruction. It does not mean that ImageNet-R has worse absolute quality at every budget or that it needs more tokens to cross an externally meaningful quality threshold.

## 4. One class-matched example

The four rows use WNID `n01443537`. The first pair reconstructs one ImageNet photograph with both models. The second pair reconstructs one ImageNet-R rendition with both models. Each model pair uses the exact same input. The photograph and rendition are different scenes from the same class.

This grid illustrates the curves; it is not a representative sample or a mechanism test. The One-D-Piece ImageNet preview was regenerated from the saved experiment checkpoint for this display. Its input hash matches the FlexTok input, and its LPIPS labels match the saved quantitative export.

In [ ]:
# Four-row comparison: two domains by two models, within one shared class.
PAIR_CLASS = "n01443537"
PAIR_BUDGETS = [8, 32, 64, 128, 256]
EXTRA_ONE_D_IMAGENET = RAW_DIR / "paired_domain_preview_v2"

def saved_assets(model, dataset):
    match = recon_examples[(recon_examples.model == model) &
                           (recon_examples.dataset == dataset) &
                           (recon_examples.wnid == PAIR_CLASS) &
                           (recon_examples.ordinal == 0)]
    assert len(match) == 1
    image_name = match.iloc[0].image
    assets = recon_manifest[(recon_manifest.model == model) &
                            (recon_manifest.dataset == dataset) &
                            (recon_manifest.image == image_name)].set_index("prefix_tokens")
    return image_name, {int(k): RECON_DIR / v for k, v in assets.asset.items()}

imagenet_image, flex_imagenet = saved_assets("flextok", "imagenet")
imagenet_r_image, flex_imagenet_r = saved_assets("flextok", "imagenet_r")
one_d_r_image, one_d_imagenet_r = saved_assets("one_d_piece", "imagenet_r")
assert one_d_r_image == imagenet_r_image
one_d_imagenet = {0: EXTRA_ONE_D_IMAGENET / f"000_{PAIR_CLASS}_original.png"}
one_d_imagenet.update({b: EXTRA_ONE_D_IMAGENET / f"000_{PAIR_CLASS}_{b:03d}_tokens.png"
                       for b in PREFIX_ORDER})
for mapping in [flex_imagenet, one_d_imagenet, flex_imagenet_r, one_d_imagenet_r]:
    assert all(mapping[b].is_file() for b in [0] + PAIR_BUDGETS)
assert hashlib.sha256(flex_imagenet[0].read_bytes()).hexdigest() == hashlib.sha256(one_d_imagenet[0].read_bytes()).hexdigest()
assert hashlib.sha256(flex_imagenet_r[0].read_bytes()).hexdigest() == hashlib.sha256(one_d_imagenet_r[0].read_bytes()).hexdigest()

rows = [
    ("ImageNet | FlexTok", "FlexTok", "imagenet", imagenet_image, flex_imagenet),
    ("ImageNet | One-D-Piece", "One-D-Piece", "imagenet", imagenet_image, one_d_imagenet),
    ("ImageNet-R | FlexTok", "FlexTok", "imagenet_r", imagenet_r_image, flex_imagenet_r),
    ("ImageNet-R | One-D-Piece", "One-D-Piece", "imagenet_r", imagenet_r_image, one_d_imagenet_r),
]
fig, axes = plt.subplots(4, len(PAIR_BUDGETS) + 1, figsize=(14, 8.8), squeeze=False)
for row_i, (label, model, dataset, image_name, assets) in enumerate(rows):
    for col_i, budget in enumerate([0] + PAIR_BUDGETS):
        ax = axes[row_i, col_i]
        with Image.open(assets[budget]) as image:
            ax.imshow(np.asarray(image), interpolation="nearest")
        ax.grid(False); ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values(): spine.set_visible(False)
        if row_i == 0:
            ax.set_title("Input" if budget == 0 else f"{budget} tokens", fontsize=11)
        if col_i == 0:
            ax.set_ylabel(label, fontsize=10, rotation=0, ha="right", va="center", labelpad=12)
        else:
            value = metric_lookup.loc[(model, dataset, PAIR_CLASS, image_name, budget), "lpips"]
            ax.set_xlabel(f"LPIPS {value:.3f}", fontsize=8, labelpad=3)
fig.suptitle(f"Class-matched reconstruction example, {PAIR_CLASS}", fontsize=14, y=.995)
fig.subplots_adjust(wspace=.05, hspace=.18, left=.17, right=.99, top=.95, bottom=.025)
save_figure(fig, "ch09_fig_class_matched_domains_and_models.png")
plt.close(fig)
display(DisplayImage(filename=str(FIGURE_DIR / "ch09_fig_class_matched_domains_and_models.png")))

**How to compare the rows.** Compare models within each domain because those rows share an exact input. Compare domains more cautiously because the two source images share a class label but not a scene. A plausible reconstructed detail need not be faithful to the input.

In [ ]:
# Build the final answers from the computed tables.
absolute_rows = []
for model in MODEL_ORDER:
    p32 = absolute_gap_df[(absolute_gap_df.model == model) &
                          (absolute_gap_df.metric == "lpips") &
                          (absolute_gap_df.prefix_tokens == 32)].iloc[0]
    p256 = absolute_gap_df[(absolute_gap_df.model == model) &
                           (absolute_gap_df.metric == "lpips") &
                           (absolute_gap_df.prefix_tokens == 256)].iloc[0]
    absolute_rows.append(
        f"| {model} | {p32.mean_r_minus_imagenet:+.4f} "
        f"[{p32.ci95_lower:+.4f}, {p32.ci95_upper:+.4f}] | "
        f"{p256.mean_r_minus_imagenet:+.4f} "
        f"[{p256.ci95_lower:+.4f}, {p256.ci95_upper:+.4f}] |"
    )

truncation_rows = []
progress_rows = []
for model in MODEL_ORDER:
    truncation = bootstrap_df[(bootstrap_df.model == model) &
                              (bootstrap_df.metric == "lpips") &
                              (bootstrap_df.prefix_tokens == 32)].iloc[0]
    truncation_rows.append(
        f"| {model} | {truncation.mean_ood_minus_imagenet:+.4f} | "
        f"[{truncation.ci95_lower:+.4f}, {truncation.ci95_upper:+.4f}] | "
        f"{round(truncation.positive_class_fraction * 200)}/200 |"
    )
    domain_values = progress_df[(progress_df.model == model) &
                                (progress_df.prefix_tokens == 32)].set_index("dataset")
    progress = progress_delta_df[(progress_delta_df.model == model) &
                                 (progress_delta_df.prefix_tokens == 32)].iloc[0]
    progress_rows.append(
        f"| {model} | {domain_values.loc['imagenet', 'fraction_achieved']:.1%} | "
        f"{domain_values.loc['imagenet_r', 'fraction_achieved']:.1%} | "
        f"{progress.imagenet_r_minus_imagenet:+.1%} "
        f"[{progress.ci95_lower:+.1%}, {progress.ci95_upper:+.1%}] |"
    )

paired = contrast_df[(contrast_df.metric == "lpips") &
                     (contrast_df.prefix_tokens == 32)].iloc[0]

text = """## What this preliminary extension found

### 1. The absolute domain comparison changes with budget

Positive values below mean higher, and therefore worse, LPIPS on ImageNet-R. The pointwise intervals directly bootstrap the class-paired domain gap.

| Model | ImageNet-R minus ImageNet at 32 | ImageNet-R minus ImageNet at 256 |
| --- | ---: | ---: |
""" + "\n".join(absolute_rows) + """

At 32 tokens, the point estimate is worse on ImageNet-R for both systems, but only FlexTok's direct interval excludes zero. At 256 tokens, ImageNet-R has better LPIPS for both systems and both intervals exclude zero. Full-budget reconstruction therefore does not summarize the domain comparison at restricted budgets. This extends Chapter 8's conclusion that reconstruction quality and representation shift answer different questions.

### 2. ImageNet-R has a larger absolute truncation cost at the planned endpoint

| Model | Adjusted LPIPS difference at 32 | Pointwise 95% interval | Positive classes |
| --- | ---: | ---: | ---: |
""" + "\n".join(truncation_rows) + f"""

Both intervals exclude zero. FlexTok's adjusted difference exceeds One-D-Piece's by **{paired.flex_minus_one_d:.4f}**, with a direct class-paired interval of **[{paired.ci95_lower:.4f}, {paired.ci95_upper:.4f}]**. This is evidence about reconstruction loss relative to each domain's 256-token endpoint. PSNR does not support the same 32-token difference, while SSIM does, so the effect remains metric-dependent.

### 3. The exploratory normalization tests curve shape

| Model | ImageNet progress at 32 | ImageNet-R progress at 32 | ImageNet-R minus ImageNet, 95% interval |
| --- | ---: | ---: | ---: |
""" + "\n".join(progress_rows) + """

This check asks whether ImageNet-R has completed less of its observed LPIPS improvement by 32 tokens after accounting for each domain's full 8-to-256 range. It should be read as an exploratory curve-shape result because the normalization was selected after the absolute analysis and depends on the chosen endpoints.

### Scope

The results support one modest conclusion: the relationship between rendition shift and reconstruction quality depends on the available prefix, the metric, and how change across the curve is defined. They do not show that individual tokens carry stable semantic concepts, prove that ordering fails, or establish that out-of-distribution images generally require more tokens. The study covers these two recorded systems, one rendition shift, one FlexTok noise realization per image, and pointwise class-bootstrap uncertainty.
"""
display(Markdown(text))